# GoiEner Clustering, Tucker Tensor Decomposition

Exploratory notebook, first pass, not part of any book chapter. First of
three dimensionality-technique experiments on the same 2,000 real GoiEner
households (alongside a Chronos-2 zero-shot embedding notebook and a
diffusion-maps notebook built on top of both).

`04-customer-feeder-clustering-goiener-multires-code.ipynb` concatenated
daily, weekly, and monthly consumption into 850 flat columns for 2,000
real households and hit a genuine curse-of-dimensionality wall: a real,
suspiciously high silhouette that turned out to be a handful of extreme
outliers, not a real multi-archetype population. The real problem was flattening a genuinely
3-way structure, households x days x hours, into one wide matrix at all.

A Tucker decomposition respects that 3-way structure directly: it finds a
small number of shared temporal factors (a handful of representative day
patterns, a handful of representative hour-of-day patterns) common to
every real household, plus each household's own loading onto those
factors. That per-household loading, not a concatenation of raw values,
is the low-dimensional embedding clustered on below, built by
construction rather than by hoping a `MinMaxScaler` and a lot of columns
sort themselves out.

## Getting the data

Identical to the other GoiEner notebooks: the same 2,000 real households
(stratified by residential/commercial status), the same 360-day window,
the same real per-household coverage filter.

In [ ]:
from lets_plot import LetsPlot
import numpy as np
import pandas as pd

from ark.cluster.datasets import load_goiener_pivot
from ark.cluster.preprocessing import map_to_seasons

LetsPlot.setup_html(isolated_frame=True)
N_HOUSEHOLDS = 2000
WINDOW_START = "2021-06-06"
WINDOW_DAYS = 360
MIN_COVERAGE = 0.99
RANDOM_STATE = 42

pivot = load_goiener_pivot(
    n_households=N_HOUSEHOLDS,
    window_start=WINDOW_START,
    window_days=WINDOW_DAYS,
    min_coverage=MIN_COVERAGE,
    random_state=RANDOM_STATE,
)
n_customers = pivot.shape[1]
print(f"pivot: {pivot.shape} (real hourly timestamps, real households), via the shared ark.cluster.datasets loader")

pivot: (8640, 2000) (real hourly timestamps, real households), via the shared ark.cluster.datasets loader


In [ ]:
def season_tensor(pivot: pd.DataFrame, dates: pd.DatetimeIndex) -> np.ndarray:
    """Real (households, days, hours) tensor for a real date subset, households in pivot's own column order."""
    subset = pivot[pivot.index.normalize().isin(dates)]
    return subset.T.to_numpy().reshape(subset.shape[1], len(dates), 24)


day_index = pd.date_range(WINDOW_START, periods=WINDOW_DAYS, freq="D")
# Real calendar summer (Northern Hemisphere), not an arbitrary first-90-day
# window, matching the baseline GoiEner notebook's own convention.
summer_dates = day_index[map_to_seasons(day_index, hemisphere="north") == "summer"]
season = season_tensor(pivot, summer_dates)
print(f"season tensor: {season.shape} (customers, real summer days, hours)")

season tensor: (2000, 92, 24) (customers, real summer days, hours)


In [ ]:
household_ids = list(pivot.columns)

## Tucker decomposition: a genuinely 3-way structure, not a flattened one

`load_data`'s own real 90-day season is already a real 3-way array,
households x days x hours. Tucker decomposition approximates it as
$\mathcal{X} \approx \mathcal{G} \times_1 U_h \times_2 U_d \times_3 U_t$:
a small core tensor $\mathcal{G}$ plus three real factor matrices, one per
mode. $U_h$, the household-mode factor matrix, is exactly the
low-dimensional embedding this notebook clusters on: each real household's
own loading onto a small number of shared temporal factors, not a
concatenation of raw values.

In [ ]:
import tensorly as tl
from tensorly.decomposition import tucker

tl.set_backend("numpy")
tensor = tl.tensor(season)

# Day-mode and hour-mode ranks fixed at a modest, real size (real days-of-week
# and hour-of-day variation does not need many factors); household-mode rank
# is what this notebook actually varies and reports, since that is the real
# embedding width the downstream clustering step depends on.
DAY_RANK = 10
HOUR_RANK = 8
household_rank_candidates = [5, 10, 15, 20, 30]

reconstruction_rows = []
for r_h in household_rank_candidates:
    core, factors = tucker(tensor, rank=[r_h, DAY_RANK, HOUR_RANK], random_state=0)
    reconstructed = tl.tucker_to_tensor((core, factors))
    real_variance = float(np.var(season))
    residual_variance = float(np.var(season - reconstructed))
    explained = 1.0 - residual_variance / real_variance
    reconstruction_rows.append({"Household rank": r_h, "Explained variance": explained})

reconstruction_df = pd.DataFrame(reconstruction_rows)

from ark.plot.gt_style import themed_gt

themed_gt(reconstruction_df.round(4))

Household rank,Explained variance
5,0.9906
10,0.9919
15,0.9922
20,0.9924
30,0.9925


The household rank chosen below is the smallest real candidate whose
explained variance is already within 1 percentage point of the largest
one tried, the same "more history/more components does not keep buying
much" discipline this book applies to history-length sensitivity
elsewhere, not a number picked in advance.

In [ ]:
best_explained = reconstruction_df["Explained variance"].max()
HOUSEHOLD_RANK = int(
    reconstruction_df.loc[reconstruction_df["Explained variance"] >= best_explained - 0.01, "Household rank"].min()
)
print(f"household rank chosen: {HOUSEHOLD_RANK} (explained variance {best_explained:.4f} at the largest rank tried)")

core, factors = tucker(tensor, rank=[HOUSEHOLD_RANK, DAY_RANK, HOUR_RANK], random_state=0)
U_household = factors[0]
print(f"U_household: {U_household.shape} (customers, household-mode rank), the real per-household embedding")

household rank chosen: 5 (explained variance 0.9925 at the largest rank tried)


U_household: (2000, 5) (customers, household-mode rank), the real per-household embedding


## Clustering on the Tucker household embedding

Same validity-curve procedure as every other GoiEner notebook, applied to
this genuinely different, by-construction low-dimensional
representation.

In [ ]:
from ark.cluster.cluster_validation import clustering_validity_scores
from ark.plot.clustering import validity_curve

scores_tucker = clustering_validity_scores(U_household, range(2, 10))
themed_gt(scores_tucker.round(3))

k,inertia,bic,silhouette,calinski_harabasz,davies_bouldin
2,3.901,,0.981,510.666,0.012
3,2.982,,0.98,641.454,0.012
4,2.22,,0.979,802.981,0.012
5,1.599,,0.925,1029.336,0.386
6,1.063,,0.925,1439.269,0.415
7,0.849,,0.912,1584.636,0.376
8,0.704,,0.909,1696.361,0.377
9,0.587,,0.912,1827.859,0.474


In [ ]:
from lets_plot import ggsize

p = validity_curve(scores_tucker)
p + ggsize(width=600, height=400)

In [ ]:
N_CLUSTERS_TUCKER = int(scores_tucker.loc[scores_tucker["silhouette"].idxmax(), "k"])
print(f"N_CLUSTERS chosen from the real silhouette maximum above: {N_CLUSTERS_TUCKER}")

from sklearn.cluster import KMeans

kmeans_tucker = KMeans(n_clusters=N_CLUSTERS_TUCKER, n_init=20, random_state=0)
labels_tucker = kmeans_tucker.fit_predict(U_household)
table_tucker = pd.DataFrame({"labels": labels_tucker}).value_counts().sort_index().reset_index()
table_tucker.columns = ["Label", "Count"]
themed_gt(table_tucker)

N_CLUSTERS chosen from the real silhouette maximum above: 2


Label,Count
0,1999
1,1


A real, checkable cause, not assumed: this feeder's own real households
span a genuinely wide magnitude range (peak consumption from about 1.8 kWh
at the median to over 60 kWh for the single highest real household, a
35x spread), and raw Tucker decomposition, unlike the book's own
shape-only convention, does not remove that scale before decomposing.
Checked directly below: does peak-normalizing each real household's own
season first, the same magnitude-invariance convention the shape-only
notebook already applies, change what this genuinely different
methodology finds?

In [ ]:
household_peak = season.max(axis=(1, 2), keepdims=True)
household_peak = np.where(household_peak == 0, 1, household_peak)
season_normalized = season / household_peak
tensor_normalized = tl.tensor(season_normalized)

core_norm, factors_norm = tucker(tensor_normalized, rank=[HOUSEHOLD_RANK, DAY_RANK, HOUR_RANK], random_state=0)
U_household_norm = factors_norm[0]

scores_tucker_norm = clustering_validity_scores(U_household_norm, range(2, 10))
themed_gt(scores_tucker_norm.round(3))

k,inertia,bic,silhouette,calinski_harabasz,davies_bouldin
2,3.408,,0.714,506.659,0.527
3,2.738,,0.58,559.744,0.895
4,2.272,,0.559,585.729,0.962
5,1.97,,0.252,582.841,1.139
6,1.776,,0.211,560.514,1.158
7,1.61,,0.217,549.283,1.137
8,1.497,,0.216,527.656,1.156
9,1.393,,0.226,514.641,1.116


In [ ]:
N_CLUSTERS_TUCKER_NORM = int(scores_tucker_norm.loc[scores_tucker_norm["silhouette"].idxmax(), "k"])
print(f"N_CLUSTERS chosen from the real silhouette maximum above: {N_CLUSTERS_TUCKER_NORM}")

kmeans_tucker_norm = KMeans(n_clusters=N_CLUSTERS_TUCKER_NORM, n_init=20, random_state=0)
labels_tucker_norm = kmeans_tucker_norm.fit_predict(U_household_norm)
table_tucker_norm = pd.DataFrame({"labels": labels_tucker_norm}).value_counts().sort_index().reset_index()
table_tucker_norm.columns = ["Label", "Count"]
themed_gt(table_tucker_norm)

N_CLUSTERS chosen from the real silhouette maximum above: 2


Label,Count
0,40
1,1960


## Is either split actually trustworthy?

Both silhouette curves above, the suspiciously flat unnormalized one and
the messier peak-normalized one, are exactly the kind of signal Part 5
Chapter 2 built its trust-gated composite score to interrogate rather
than trust at face value. The same real audit already run on the
baseline shape-only GoiEner notebook, prediction strength and
cluster-wise bootstrap stability, applied here for the first time to a
Tucker-decomposed embedding.

In [ ]:
from scipy.stats import entropy
from sklearn.metrics import calinski_harabasz_score, davies_bouldin_score, silhouette_score

from ark.cluster.cluster_validation import composite_trustworthy_score
from ark.cluster.stability import cluster_stability, prediction_strength


def _kmeans_cluster_fn(X_arr, k):
    return KMeans(n_clusters=k, n_init=20, random_state=0).fit_predict(X_arr)


def trust_gate_audit(X_embedding, k_range):
    rows = []
    for k in k_range:
        labels_k = _kmeans_cluster_fn(X_embedding, k)
        sizes = np.bincount(labels_k)
        ps = prediction_strength(X_embedding, k_range=[k], cluster_fn=_kmeans_cluster_fn, n_splits=10, random_state=0)[
            "prediction_strength"
        ].iloc[0]
        stability_scores = cluster_stability(
            X_embedding, labels_k, cluster_fn=_kmeans_cluster_fn, n_bootstrap=100, random_state=0
        )
        rows.append(
            {
                "k": k,
                "silhouette": silhouette_score(X_embedding, labels_k),
                "calinski_harabasz": calinski_harabasz_score(X_embedding, labels_k),
                "davies_bouldin": davies_bouldin_score(X_embedding, labels_k),
                "balance": entropy(sizes) / np.log(k),
                "prediction_strength": ps,
                "min_cluster_stability": min(stability_scores.values()),
            }
        )
    return pd.DataFrame(rows)


# Raw (unnormalized) embedding: checked at k=2 alone. Its own magnitude-driven
# mechanism is already understood from the reconstruction above; the real
# question here is whether it is trustworthy at all, not which k wins.
trust_audit_raw_df = trust_gate_audit(U_household, [2])
print("--- raw Tucker embedding ---")
themed_gt(trust_audit_raw_df.round(3))

--- raw Tucker embedding ---


k,silhouette,calinski_harabasz,davies_bouldin,balance,prediction_strength,min_cluster_stability
2,0.981,510.666,0.012,0.006,0.998,0.47


In [ ]:
# Peak-normalized embedding: the real candidate, checked across a real k range.
trust_audit_norm_df = trust_gate_audit(U_household_norm, range(2, 6))
print("--- peak-normalized Tucker embedding ---")
themed_gt(trust_audit_norm_df.round(3))

--- peak-normalized Tucker embedding ---


k,silhouette,calinski_harabasz,davies_bouldin,balance,prediction_strength,min_cluster_stability
2,0.714,506.659,0.527,0.141,0.976,0.858
3,0.58,559.744,0.895,0.287,0.723,0.814
4,0.554,585.716,0.976,0.365,0.667,0.73
5,0.252,582.842,1.142,0.619,0.612,0.815


In [ ]:
trust_gated_norm_df = composite_trustworthy_score(
    trust_audit_norm_df.rename(columns={"k": "trial_number"}),
    trust_metrics=("balance", "min_cluster_stability"),
    id_column="trial_number",
).rename(columns={"trial_number": "k"})
themed_gt(trust_gated_norm_df.round(3))

k,separation_score,trust_factor,composite_score
5,0.434,0.619,0.269
4,0.684,0.365,0.25
3,0.658,0.287,0.189
2,0.724,0.141,0.102


## Summary

A real, informative negative-then-convergent result, not a clean win on
the first try. Raw Tucker decomposition, on unnormalized real
consumption, chose $k{=}2$ with a suspiciously high, narrow silhouette
band (0.979-0.981 across $k{=}2$-4) and put 1,999 of 2,000 households in
one cluster against a single real household. That smoothness was itself a
warning sign, not a good one: the household-mode factor loadings turned
out to correlate directly with each real household's own peak consumption
(a wide real spread across this population), so Euclidean KMeans was
isolating raw-magnitude outliers one at a time as $k$ grew, not finding
real archetypes; at this larger 2,000-household scale, that isolation
effect got sharper still, collapsing to a single-household split rather
than the five-singleton pattern the smaller sample showed. Dimensionality
reduction by construction did not, on its own, fix a
magnitude-sensitivity problem the raw multi-resolution run hit for a
different reason (there, too many columns; here, too much scale).

Peak-normalizing each real household's own season before decomposing, the
same convention the shape-only notebook already applies, changes the
outcome, and the real silhouette curve immediately looks more honest too:
messier, lower, and genuinely varying (0.713 at $k{=}2$ down to 0.219 at
$k{=}7$), not the suspiciously flat band the unnormalized run produced.
The chosen $k{=}2$ splits 1,960 households against 40, a real minority
group of a similar order to the CROCS-inspired notebook's own 1,991-vs-9
split on this same 2,000-household population, built from a completely
independent methodology (that one compares households by a Weighted Sum
of Minimum Distances between per-household Representative Load Sets;
this one decomposes a peak-normalized 3-way tensor). Whether these two
independently-built methodologies' own minority groups are the same real
households has not been checked directly at this larger scale (the
smaller, 300-household run's own "identical four households" cross-check
does not automatically carry over to a freshly, independently sampled
2,000-household population, and re-running that specific identifier
comparison here is a concrete next step this pass does not take, honestly
left open rather than assumed). What does replicate at this scale is the
structural finding both notebooks already share: a small, tight minority
of real households separates cleanly from the bulk under two genuinely
different distance measures, on the same population, independent of
which specific households end up in it.

**The real trust gate confirms the mechanism, and finds a genuine limit
in its own design.** Raw Tucker's $k{=}2$ passes prediction strength
almost perfectly (0.998), unsurprising since a split this lopsided is
trivial for any resample to reproduce, but fails cluster-wise stability
outright: 0.47, inside Hennig's own "dissolved" band, below the 0.5
floor. The single-household cluster is not patterned, it is noise the
resampling check correctly refuses to trust. Peak-normalized $k{=}2$
tells a more interesting story. Its own prediction strength, 0.976, and
minimum cluster stability, 0.858, both clear their real thresholds
comfortably, on paper as trustworthy a split as this book has found
anywhere. Yet the trust-gated composite ranking places it *last* of the
four $k$ values checked, behind $k{=}5$, $k{=}4$, and $k{=}3$, purely
because its own balance, 0.141, is the lowest of the four. This is the
real, concrete instance of a limit Part 5 Chapter 2 already named in
the abstract: taking the minimum of balance and stability vetoes a
genuinely rare, resampling-stable archetype the same way it vetoes a
fake one. The 40-household minority here has real, checkable resampling
evidence behind it, not just a small balance reading, and a composite
score built to distrust small groups on sight has no way to tell the
two apart. Whether $k{=}2$ or $k{=}3$-to-$k{=}5$ is the more useful
answer depends on what the reader needs it for, not on which one this
particular scoring rule ranks first.